# validationで固定した動作点

共通閾値0のEO指標と、ERMの群別閾値参照を分ける。testで閾値を最適化しない。

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import torch
from omegaconf import OmegaConf

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import local_data_path  # noqa: E402
from analysis.common.predictions import load_cache, stale_reason, write_cache  # noqa: E402
from analysis.archive.minimax_auc_vs_eo.evaluation import (  # noqa: E402
    age_metrics,
)

STUDY = ROOT / "analysis/archive/minimax_auc_vs_eo"
RESULTS = STUDY / "results"
RESULTS.mkdir(exist_ok=True)
RUNS = [
    path
    for path in sorted((STUDY / "runs").glob("*"))
    if (path / "run.json").is_file() and json.loads((path / "run.json").read_text()).get("status") == "succeeded"
]

## 明示checkpointの予測読み込み

In [ ]:
def imported_checkpoint(run, row):
    if row["epoch"] > 0:
        return run, Path(row["checkpoint"])
    parent_id = json.loads((run / "run.json").read_text())["parent_run"]["run_id"]
    parent_run = STUDY / "runs" / parent_id
    if not (parent_run / "run.json").is_file():
        raise ValueError(f"親runをstudyへ取り込む必要がある: {parent_id}")
    return parent_run, Path("checkpoints") / Path(row["checkpoint"]).name


def predictions(run, row, split):
    source, checkpoint = imported_checkpoint(run, row)
    matches = list((STUDY / "cache").glob(f"{source.name}_*_{split}.npz"))
    source_hashes = {}
    path = next(
        (
            path
            for path in matches
            if stale_reason(path, source, checkpoint=checkpoint, source_hashes=source_hashes) is None
        ),
        None,
    )
    if path is None:
        path = write_cache(
            source,
            split,
            STUDY / "cache",
            torch.device("cuda" if torch.cuda.is_available() else "cpu"),
            checkpoint=checkpoint,
            source_hashes=source_hashes,
        )[0]
    config = OmegaConf.load(source / "config.yaml")
    frame = pd.read_csv(local_data_path(config.data.cv_splits_dir) / f"{split}.csv")
    if frame["age_missing"].astype(bool).any():
        raise ValueError("年齢欠損がある")
    cache = load_cache(path, frame.image.to_numpy(dtype=str))
    return (
        cache["logits"][:, 1] - cache["logits"][:, 0],
        cache["target"],
        (frame.age.to_numpy() >= 65).astype(int),
        frame,
    )

In [ ]:
selection = json.loads((RESULTS / "selection.json").read_text()) if (RESULTS / "selection.json").exists() else {}
final_ids = (
    json.loads((RESULTS / "final_run_ids.json").read_text()) if (RESULTS / "final_run_ids.json").exists() else []
)
TARGET_TPR = 0.8  # test を読む前に固定する参照目標
rows = []
seen_parents = set()
for run_id in final_ids:
    run = STUDY / "runs" / run_id
    # 群別閾値参照は親 ERM についてのみ求める。
    parent = selection[run_id]["parent"]
    parent_run, parent_checkpoint = imported_checkpoint(run, parent)
    parent_key = (parent_run.name, str(parent_checkpoint))
    if parent_key in seen_parents:
        continue
    seen_parents.add(parent_key)
    experiment_seed = int(OmegaConf.load(run / "config.yaml").seed)
    score, y, g, _ = predictions(run, parent, "val")
    thresholds = [
        float(np.quantile(score[(g == group) & (y == 1)], 1 - TARGET_TPR, method="lower")) for group in range(2)
    ]
    test_score, test_y, test_g, _ = predictions(run, parent, "test")
    probability = (test_score > np.array(thresholds)[test_g]).astype(float)
    common = age_metrics(test_score, test_y, test_g)
    rows.append(
        dict(
            run_id=run_id,
            parent_run_id=parent_run.name,
            seed=experiment_seed,
            label="parent_erm_common_threshold_0",
            target_validation_tpr=TARGET_TPR,
            **common,
        )
    )
    row = age_metrics(test_score, test_y, test_g, positive_probability=probability)
    rows.append(
        dict(
            run_id=run_id,
            parent_run_id=parent_run.name,
            seed=experiment_seed,
            label="parent_erm_group_threshold_reference",
            thresholds=thresholds,
            target_validation_tpr=TARGET_TPR,
            **row,
        )
    )
if rows:
    pd.DataFrame(rows).to_csv(RESULTS / "operating_points.csv", index=False)
print(f"動作点を評価したrun: {len(rows)}")

## 解釈

群別閾値はモデル再学習と異なる参照条件。目標TPRは量子化・tieにより厳密に一致しないため、達成率も報告する。学習時の共通閾値0の結果とは別に読む。

## 共通閾値と群別閾値参照の動作点

親ERMの共通閾値0と群別閾値を並べ、各年齢群のTPR・TNRを示す。群別閾値はvalidationだけで決定した参照条件。

In [ ]:
if rows:
    table = pd.DataFrame(rows)
    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    conditions = "seed " + table.seed.astype(str) + " / " + table.label.str.replace("parent_erm_", "", regex=False)
    positions = np.arange(len(table))
    for axis, metric in zip(axes, ("tpr", "tnr"), strict=True):
        for group, label in enumerate(("<65", "≥65")):
            axis.plot(positions, [value[group] for value in table[metric]], marker="o", label=label)
        axis.set_title(metric.upper())
        axis.set_ylim(0, 1)
        axis.set_xticks(positions, conditions, rotation=60, ha="right")
        axis.grid(alpha=0.2)
        axis.legend()
    figure.tight_layout()
    (STUDY / "figures").mkdir(exist_ok=True)
    figure.savefig(STUDY / "figures/operating_points.png", dpi=200)
    plt.show()